# Mobile apps review classification with spaCy and sklearn (Vector Space Model and Logistic Regression)

This notebook illustrates text classification using **Vector Space Model (VSM)** for text representation and **Logistic Regression (LR)** algorithm for classification. <br>

The classification task is classification of mobile app reviews into:
* functional requirements,
* non-functional requirements,
* other (reviews irrelevant from Requirements Engineering perspective)

The dataset used in the notebook originates from the paper [Listening to the Crowd for the Release Planning of Mobile Apps](https://dl.acm.org/doi/abs/10.1109/TSE.2017.2759112). It consists of manually classified 3,000 user reviews of different Android apps.<br>

### Load the required libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn import metrics

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from IPython.display import display

import re
import spacy

import json
from pathlib import Path

In [ ]:
RAND_STATE = 1

### Load the data

In [ ]:
data = pd.read_csv(Path.cwd() / 'data' / 'mobile_apps_reviews.csv')

Loading the data in case of running this code from Google Colab

In [ ]:
# from google.colab import files
#
# data_file = files.upload()
#
# The uploaded object is a dictionary with the filename and the file content as key-value pairs - in this case just one such pair.
# We extract the file name and read in the file content using pandas (to have it in the DataFrame format)
# file_name = list(data_file.keys())[0]
# data = pd.read_csv(file_name)

In [ ]:
data.head(10)

In [ ]:
data.info()

Create the outcome variable (`cls`) as was done in the previous class:
* functional: BUG, FEATURE
* non-functional: PERFORMANCE, USABILITY, ENERGY, SECURITY
* other: OTHER

In [ ]:
def category_to_cls(cat):
  if cat == "OTHER": return cat
  if cat in ['BUG', 'FEATURE']: return "FUNCTIONAL"
  return "NON-FUNCTIONAL"

data['cls'] = data.category.apply(category_to_cls)
data.cls.value_counts(normalize=True)

As at the previous class, we will include review length and rating as predictors, in addition to the review text. The other variables will be dropped

In [ ]:
data['review_len'] = data.review.apply(len)

In [ ]:
data_sub = data[['rating', 'review_len', 'review', 'cls']]
data_sub.head(10)

### Text cleaning (optional)

**Note**: to reduce the complexity of this task, we may skip the text cleaning component as it is not necessary for the text we are currently working with. Still, it is an often used component when working with text collected from the Web, and you are encouraged to explore it.

If (textual) data was collected from web sites, forums, online social networks, and other similar sources, it often needs to be "cleaned" before being sent to spaCy for preprocessing. This cleaning typically includes removal of HTML markup,links, hashtags and the like.

In [ ]:
def clean_text(text):
  text = re.sub(r"(?:^|\s)#(\w+)", " \\1", text) # remove # from hashtags
  text = re.sub(r"(?:^|\s)@(\w+)", "", text) # remove any refences to other users
  text = re.sub("<[^>]+>", "", text) # remove HTML mark-up (if any)
  text = text.strip()
  return text

In [ ]:
print(clean_text("love my #dragon have #fun"))
print(clean_text("this is the best - recommend to @all"))
print(clean_text("<b>don't like it at all</b>"))

For a refresher on regular expressions, check a cheatsheet such as [this one](https://www.dataquest.io/cheat-sheet/regular-expressions-cheat-sheet/).

To be able to include text cleaning component in the model building pipeline, we will create a custom transformer by subclassing scikit-learn's base class for all transformers ([`TransformerMixin`](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html)). This is mandatory for integrating it into a sklearn Pipeline (only transformers and estimators can be included).<br> This will be the first step in text preprocessing (and the overall pipeline), which will be followed by the 'vectorisation' step.

In [ ]:
class TextCleaner(TransformerMixin):

    def transform(self, X):
        X['review'] = [clean_text(txt) for txt in X['review']]
        return X

    def fit(self, X, y=None):
        return self


### Text transformation with the Vector Space Model

#### Text preprocesing (using Spacy)

First, we need to prepare a function for text preprocessing. This will be done in the same way as in the previous class - using spacy.

So, we star by loading spacy's basic English language model, that is, a pipeline of software componets for processing English language texts.

In [ ]:
nlp_pipe = spacy.load('en_core_web_sm', exclude=['parser','ner'])

Reminder: if the previous line ends in an error, it is because spacy's `en_core_web_sm` language model is not available. <br>To download it, execute the following line in the command prompt or terminal: <br>
`python -m spacy download en_core_web_sm` <br>

We will have the same function as before for custom preprocessing of app reviews, using spacy's pipeline:

In [ ]:
def spacy_preprocessing(doc):
    # Create tokens object with linguistic annotations
    tokens = nlp_pipe(doc)

    # remove punctuation, numbers, and URLs
    tokens = [token for token in tokens if (not token.is_punct) and (not token.is_space) and (not token.like_num) and (not token.like_url)]

    # remove stopwords
    tokens = [token for token in tokens if (not token.is_stop) or (token.lemma_.lower() == 'not')]

    # Lemmatize each token and convert each token into lowercase
    words = [token.lemma_.lower() for token in tokens]

    # return preprocessed list of words
    return words

#### Text transformation with TF-IDF vectoriser

Next, we create TF-IDF vectoriser that, for the given set of documents (reviews), builds a vector space model with TF-IDF weights.

This vectoriser is an instance of the scikit-learn's [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) class

In [ ]:
def create_tfidf_vectoriser(ngrams=(1,2), min_doc_freq=0.01):

    tfidf_vectorizer = TfidfVectorizer(
        tokenizer = spacy_preprocessing, # note the use of the above defined spacy_preprocessing f.
        token_pattern=None, #since we are using a custom tokeniser, we do not use token matching of TF-IDF vectoriser
        ngram_range = ngrams,
        min_df = min_doc_freq)

    return tfidf_vectorizer

We can examine what the transofmed data looks like

In [ ]:
vectoriser = create_tfidf_vectoriser()
vectorised_reviews = vectoriser.fit_transform(data_sub.review)
df = pd.DataFrame(data=vectorised_reviews.toarray(),
                  columns=vectoriser.get_feature_names_out())

display(df.shape)
display(df.iloc[:10,:10])

### Create and fit text classification pipeline

As before, we will create a review classifier as a sequence ('pipeline') of several components, using the [sklearn's Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

To build a Pipeline, first, we split the data into train/test sets:

In [ ]:
X = data_sub[['review','review_len','rating']]
y = data_sub.cls.values

# do label encoding
y_encodings, y_levels = pd.factorize(y)

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y_encodings,
                                                    test_size = 0.2,
                                                    random_state = RAND_STATE,
                                                    stratify=y_encodings)

As before, we use the [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) class to apply different data transformations to different columns in the dataset.

In [ ]:
# Instantiate Tf-Idf vectoriser
tf_idf_vectorizer = create_tfidf_vectoriser()

# we need to have all the features in a similar value range,
# so, we apply a scaler to the review length and rating features
scaler = RobustScaler()

# Apply Tf-Idf vectoriser to the 'review' column and add the other two columns (rating and review length) as is
column_transformer = ColumnTransformer(
    [('tf_idf', tf_idf_vectorizer, 'review'),
     ('scaler', scaler, ['review_len','rating'])]
)

As the classifier, we will use **Logistic Regression (LR)** classifier (implemented in sklearn's [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) class).

In [ ]:
classifier = LogisticRegression(random_state=RAND_STATE,
                                max_iter=2000)

We can now define and fit the classification pipeline:

In [ ]:
pipe = Pipeline([('cleaner', TextCleaner()),
                 ('vectorizer', column_transformer),
                 ('classifier', classifier)])

pipe.fit(X_train, y_train)

### Evaluate the model on the test data

First, examine the confusion matrix

In [ ]:
disp = metrics.ConfusionMatrixDisplay.from_estimator(
        pipe,
        X_test,
        y_test,
        display_labels=y_levels,
        cmap='Blues',
    )

Now, we can compute evaluation measures:

In [ ]:
predictions_1 = pipe.predict(X_test)

report_1 = metrics.classification_report(y_true = y_test, y_pred = predictions_1)
print(report_1)

Reminder:

*   `support` is the number of occurrences of each class in y_true
*   `macro avg` is the average of unweighted score per label (i.e. class)
*   `weighted avg` is the average of support-weighted score per label (i.e. class):

weighted avg for metric $m$:

$m = \sum_{i=1}^{n} m_i * \frac{support_i}{N}$

You may want to take a look at [this](https://www.evidentlyai.com/classification-metrics/multi-class-metrics) or [this](https://iamirmasoud.com/2022/06/19/understanding-micro-macro-and-weighted-averages-for-scikit-learn-metrics-in-multi-class-classification-with-example/) article for an excellent explanation of how all these different metrics are computed and when they should be used.

Compute also [balanced accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html).

Reminder: balanced accuracy is, in fact, raw accuracy where each sample is weighted according to the inverse prevalence (see the "support" in the table above) of its true class.

In [ ]:
# balanced accuracy, adjusted for chance
display(f"Accuracy: {metrics.accuracy_score(y_test, predictions_1):0.4f}" )
display(f"Balanced accuracy: {metrics.balanced_accuracy_score(y_test, predictions_1, adjusted=True):0.4f}" )

### Try to improve the classifer through hyperparameter tuning

We will now try to improve the performance of the classifier by changing the way text transformation is done and how LR algorithm is configured.

Hyperparameters often used for tuning TF-IDF vectorise include:
* `min_df` - minimum document freequency
* `ngram_range` - ngrams that will be used as features
* `max_features` - the nuber of ngrams to keep, sorted based on their TF-IDF values

When it comes to Logistic Regression in sklearn, there are a couple of important hyperparameters (C, penalty, solver). Among these, most often tuned one is `C` (Inverse Regularization Strength), which controls the trade-off between fitting the training data and avoiding overfitting:
* Low C values lead to a stronger regularization. It shrinks coefficients and creates a simpler model (higher bias, lower variance)
* Larger C values imply weaker regularization. It leads to larger coefficients, fitting training data more closely (lower bias, higher variance)

We will try to determine optimal value of the above hyperparameters through *Cross-Validation*, using the [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) class.

**Reminder**: hyperparamter names have to be preceded by the name of the corresponding component (transformer or estimator) in the pipeline; in this case, names of the relevant pipeline components are `vectorizer` and `classifier`. However, as the 'vectorizer' is not a single transformer but may contain multiple transformers (one per column), we need to specify the name of the particular transformer we want to fine-tune - in this case, it is 'word_counts'. So, we have to precede the name of the text transformation hyperparameter with `vectorizer__word_counts__`.

In [ ]:
%%time

param_distributions = {
    'vectorizer__tf_idf__min_df': [0.005, 0.01, 0.025, 0.05],
    'classifier__C': np.logspace(-3, 3, 7) #[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}

random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=10,          # randomly try exactly 10 distinct combinations
    scoring='f1_macro',
    random_state=RAND_STATE,
    verbose=2
)

random_search.fit(X_train, y_train)

Note: `logspace` is a NumPy function that generates numbers that are evenly spaced on a logarithmic scale, rather than a linear scale. For example, for the log base of 10, `logspace` multiplies each value by 10 to produce the next one; so, each step leads to an order of magnitude increase.

In [ ]:
print("Best Hyperparameters Found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Macro F1: {random_search.best_score_:.4f}")

Re-run the pipeline with the optimal hyperparamter values:

In [ ]:
best_min_df = random_search.best_params_['vectorizer__tf_idf__min_df']
best_C = random_search.best_params_['classifier__C']

# note how we configure the pipeline with a specific value for a hyperparameter
pipe.set_params(vectorizer__tf_idf__min_df=best_min_df, classifier__C=best_C)
pipe.fit(X_train, y_train)

Make predictions with the fine-tuned pipeline and evaluate the performance

In [ ]:
predictions_2 = pipe.predict(X_test)

report_2 = metrics.classification_report(y_true=y_test, y_pred=predictions_2)
print(report_2)

In [ ]:
print(f"Balanced accuracy: {metrics.balanced_accuracy_score(y_test, predictions_2, adjusted=True):0.4f}" )

### Compare the created models

We will now compare the models we have built so far and will also include a dummy classifier to see how much better the models are compared to the most basic classifier (the one that generates predictions by respecting the training set’s class distribution)

In [ ]:
dummy_clf = DummyClassifier(strategy='stratified') # generates predictions by respecting the training set’s class distribution

dummy_clf.fit(X_train, y_train)
dummy_pred = dummy_clf.predict(X_test)
report_dummy = metrics.classification_report(y_true = y_test, y_pred = dummy_pred)
print(report_dummy)

The following function calculates macro and weighted average metrics and balanced accuracy based on the given input paramters for one classifier

In [ ]:
def get_model_metrics(y_true:np.ndarray, y_pred:np.ndarray, model_name:str):

    # compute P, R, F1 for both averages
    report = metrics.classification_report(y_true, y_pred, output_dict=True)

    # compute balanced accuracy
    b_acc = metrics.balanced_accuracy_score(y_true, y_pred)

    # extract metrics for the specified averages
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']

    results = {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': b_acc
    }

    return results

In [ ]:
def load_model_metrics_from_json(fpath:str, model_name:str):

    try:
        with open(fpath, 'r') as fobj:
            scores_dict = json.load(fobj)
    except OSError as err:
        print(f"Error while trying to read evaluation results for {model_name} from {fpath}")
        print(err)
        return None

    # extract metrics for the specified averages
    macro_avg = scores_dict['macro avg']
    weighted_avg = scores_dict['weighted avg']

    results = {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': None
    }

    return results

In [ ]:
model_predictions = {
    'TF-IDF_LR': predictions_1,
    'TF-IDF_LR_FT': predictions_2,
    'DUMMY': dummy_pred
}

all_results = list()

for model_name, model_pred in model_predictions.items():
  model_res = get_model_metrics(y_test, model_pred, model_name)
  all_results.append(model_res)


bow_results = load_model_metrics_from_json(Path.cwd() / 'results' / 'BoW_best_model_scores.json', 'BOW_DT_FT')
if bow_results:
    all_results.append(bow_results)

all_results_df = pd.DataFrame(all_results)
all_results_df.set_index("Model", inplace=True)
all_results_df = all_results_df.apply(lambda x: round(x, 4))

print("Model Performance Comparison:")
print(all_results_df)